In [1]:
# Đọc history các train node và kết luận chọn một ngày

In [1]:
import pandas as pd
import json
from sklearn.metrics.pairwise import haversine_distances
import numpy as np

# Load train df

In [2]:
nodes_df = pd.read_csv("../data/raw/nodes.csv")
nodes_df.head()

,_id,long,lat
0,366367223,106.629056,10.804243
1,366367233,106.709701,10.771110
2,366367242,106.737189,10.709337
3,366367274,106.760081,10.854489
4,366367285,106.721163,10.804994


In [3]:
train_df = pd.read_csv("../data/raw/train.csv")
train_df.head()

,_id,segment_id,date,weekday,period,LOS,s_node_id,e_node_id,length,street_id,max_velocity,street_level,street_name,street_type,long_snode,lat_snode,long_enode,lat_enode
0,0,26,2021-04-16,4,period_0_30,A,366428456,366416066,116,32575820,NaN,4,Nguyễn Văn Bá,tertiary,106.768732,10.841506,106.769254,10.842422
1,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,26,32575862,NaN,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808
2,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,26,32575862,NaN,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808
3,3,67,2021-03-09,1,period_9_30,B,366403668,5755066033,7,32575862,NaN,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771
4,4,67,2021-03-23,1,period_9_30,B,366403668,5755066033,7,32575862,NaN,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771


# Đọc file json

In [4]:
with open("../data/raw/osm_full_history_all.json", "r", encoding="utf-8") as f:
    node_history = json.load(f)

In [5]:
node_history.keys()

dict_keys(['elements', 'meta'])

In [6]:
node_history["meta"]

{'total_nodes': 11312, 'total_versions': 37247, 'source_files': 1132}

# Load history df

In [7]:
node_elements = node_history["elements"]
node_elements[0]

{'type': 'node',
 'id': 366367392,
 'lat': 10.7760293,
 'lon': 106.6141472,
 'timestamp': '2009-03-27T23:04:12Z',
 'version': 1,
 'changeset': 863232,
 'user': 'Kapis',
 'uid': 94270}

In [8]:
node_history_df = pd.json_normalize(node_elements).sort_values("id")
print(node_history_df.shape)
node_history_df.head()

(37247, 89)


,type,id,lat,lon,timestamp,version,changeset,user,uid,tags.crossing,...,tags.diet:vegan,tags.addr:city,tags.addr:district,tags.tourism,tags.access:conditional,tags.surface,tags.wheelchair,tags.addr:subdistrict,tags.alt_name,tags.flashing_lights
0,node,366367392,10.776029,106.614147,2009-03-27T23:04:12Z,1,863232,Kapis,94270,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,366367392,10.775756,106.613980,2013-06-18T03:08:31Z,2,16598250,QuangDBui@TMA,1276367,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,366367392,10.775732,106.614032,2018-07-19T07:28:17Z,3,60859406,neela1,7795604,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,366367392,10.775755,106.614047,2019-07-19T13:11:55Z,4,72430889,szilardk_grab,10143877,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,366367451,10.793771,106.695444,2011-11-25T17:55:36Z,2,9946833,Dymo12,509465,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Chọn mốc thời gian phù hợp

In [9]:
train_node_ids = (
    set(train_df["s_node_id"]) | 
    set(train_df["e_node_id"])
)

In [10]:
filtered_nodes_df = nodes_df[nodes_df["_id"].isin(train_node_ids)]

In [11]:
train_node_locs = (
    set(
        train_df[train_df["s_node_id"].isin(train_node_ids)]
            [["s_node_id", "long_snode", "lat_snode"]]
            .apply(tuple, axis=1)
    ) |
    set(
        train_df[train_df["e_node_id"].isin(train_node_ids)]
            [["e_node_id", "long_enode", "lat_enode"]]
            .apply(tuple, axis=1)
    )
)

In [12]:
osm_node_locs = set(node_history_df[["id", "lon", "lat"]].apply(tuple, axis=1))

In [13]:
inter_node_locs = train_node_locs & osm_node_locs

In [14]:
print(len(osm_node_locs))
print(len(train_node_locs))
print(len(inter_node_locs))

34358
11314
11312


In [15]:
mismatch_node_ids = [x[0] for x in list(train_node_locs - osm_node_locs)]
nodes_df[nodes_df["_id"].isin(mismatch_node_ids)]

,_id,long,lat
558635,5806315850,106.678132,10.757173
558675,5806323052,106.753776,10.678259


In [16]:
# Tìm mốc thời gian bắt đầu dữ liệu thiếu nhất quán

In [17]:
temp_history_df = node_history_df[["id", "lon", "lat", "timestamp", "version"]].copy()
temp_history_df = temp_history_df.rename(columns={
    "lon": "osm_long",
    "lat": "osm_lat"
})
print(temp_history_df.shape)
temp_history_df.head()

(37247, 5)


,id,osm_long,osm_lat,timestamp,version
0,366367392,106.614147,10.776029,2009-03-27T23:04:12Z,1
1,366367392,106.613980,10.775756,2013-06-18T03:08:31Z,2
2,366367392,106.614032,10.775732,2018-07-19T07:28:17Z,3
3,366367392,106.614047,10.775755,2019-07-19T13:11:55Z,4
4,366367451,106.695444,10.793771,2011-11-25T17:55:36Z,2


In [18]:
temp_train_df = filtered_nodes_df.copy()
temp_train_df = temp_train_df.rename(columns={
    "_id": "id",
    "long": "train_long",
    "lat": "train_lat"
})
print(temp_train_df.shape)
temp_train_df.head()

(11314, 3)


,id,train_long,train_lat
17,366367392,106.614032,10.775732
25,366367451,106.695366,10.793792
56,366367839,106.664528,10.807689
57,366367847,106.671636,10.814589
65,366368010,106.644773,10.763042


In [19]:
combine_df = temp_train_df.merge(
    temp_history_df,
    how="inner",
    on="id"
)

# Bỏ node hoàn toàn không trùng
combine_df = combine_df[
    ~combine_df["id"]
    .isin(mismatch_node_ids)
].reset_index()

print(combine_df.shape)
combine_df.head()

(37247, 8)


,index,id,train_long,train_lat,osm_long,osm_lat,timestamp,version
0,0,366367392,106.614032,10.775732,106.614147,10.776029,2009-03-27T23:04:12Z,1
1,1,366367392,106.614032,10.775732,106.613980,10.775756,2013-06-18T03:08:31Z,2
2,2,366367392,106.614032,10.775732,106.614032,10.775732,2018-07-19T07:28:17Z,3
3,3,366367392,106.614032,10.775732,106.614047,10.775755,2019-07-19T13:11:55Z,4
4,4,366367451,106.695366,10.793792,106.695444,10.793771,2011-11-25T17:55:36Z,2


In [20]:
train_coords = np.radians(
    combine_df[["train_lat", "train_long"]].values
)

osm_coords = np.radians(
    combine_df[["osm_lat", "osm_long"]].values
)

earth_radius_m = 6371000

combine_df["dist"] = np.array([
    haversine_distances(
        [train_coords[i]],
        [osm_coords[i]]
    )[0, 0] * earth_radius_m
    for i in range(len(combine_df))
])

In [21]:
combine_df.head()

,index,id,train_long,train_lat,osm_long,osm_lat,timestamp,version,dist
0,0,366367392,106.614032,10.775732,106.614147,10.776029,2009-03-27T23:04:12Z,1,35.363190
1,1,366367392,106.614032,10.775732,106.613980,10.775756,2013-06-18T03:08:31Z,2,6.246626
2,2,366367392,106.614032,10.775732,106.614032,10.775732,2018-07-19T07:28:17Z,3,0.000000
3,3,366367392,106.614032,10.775732,106.614047,10.775755,2019-07-19T13:11:55Z,4,3.070430
4,4,366367451,106.695366,10.793792,106.695444,10.793771,2011-11-25T17:55:36Z,2,8.783031


In [22]:
# ====================== PIVOT FIX - PHIÊN BẢN AN TOÀN ======================
combine_df['timestamp'] = pd.to_datetime(combine_df['timestamp'], errors='coerce')

# Bỏ timezone để tránh lỗi Grouper
combine_df['timestamp'] = combine_df['timestamp'].dt.tz_localize(None)

# Tạo cột date riêng (để dễ debug)
combine_df['date'] = combine_df['timestamp'].dt.date

print("Số rows sau khi xử lý:", len(combine_df))
print(combine_df['date'].value_counts().sort_index().head(10))

# === Pivot cách 1: Dùng cột date (ổn định nhất) ===
pivot = combine_df.pivot_table(
    index='date',           # Dùng cột date thay vì Grouper
    columns='id',
    values='dist',
    aggfunc='first'         # thay bằng 'min' nếu muốn khoảng cách nhỏ nhất
)

# Tạo full date range
min_date = combine_df['date'].min()
max_date = combine_df['date'].max()

full_range = pd.date_range(start=min_date, end=max_date, freq='D')

pivot = pivot.reindex(full_range)

# ====================== KIỂM TRA ======================
print("\nShape pivot:", pivot.shape)
print("Tổng NaN:", pivot.isna().sum().sum())
print("Tỉ lệ NaN (%):", round(pivot.isna().sum().sum() / pivot.size * 100, 2))

Số rows sau khi xử lý: 37247
date
2009-03-27    159
2009-03-28    628
2009-03-30      1
2009-04-04      4
2009-04-08      3
2009-04-16      1
2009-04-24      1
2009-05-17      2
2009-05-18      3
2009-05-26      1
Name: count, dtype: int64

Shape pivot: (6262, 11312)
Tổng NaN: 70801469
Tỉ lệ NaN (%): 99.95


In [23]:
# Bỏ các ngày hoàn toàn null
active_df = pivot.dropna(how="all")
print(active_df.shape)

(2060, 11312)


In [24]:
active_df = active_df.ffill().bfill()
active_df.head()

id,366367392,366367451,366367839,366367847,366368010,366368046,366368048,366368058,366368110,366368118,...,6175905346,6175905349,6175905353,6175931648,6175931660,6175931663,6175931668,6175932025,6175932032,6175932046
2009-03-27,35.36319,8.783031,6.961568,14.403596,7.47386,48.95393,7.847527,4.39975,8.06361,1.72066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2009-03-28,35.36319,8.783031,6.961568,14.403596,7.47386,48.95393,7.847527,4.39975,8.06361,1.72066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2009-03-30,35.36319,8.783031,6.961568,14.403596,7.47386,48.95393,7.847527,4.39975,8.06361,1.72066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2009-04-04,35.36319,8.783031,6.961568,14.403596,7.47386,48.95393,7.847527,4.39975,8.06361,1.72066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2009-04-08,35.36319,8.783031,6.961568,14.403596,7.47386,48.95393,7.847527,4.39975,8.06361,1.72066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
mean_active_df = active_df.mean(axis=1)

In [26]:
mean_active_df[mean_active_df == mean_active_df.min()]

2019-01-02    0.359649
dtype: float64

In [32]:
bool_df = active_df <= 1e-8
bool_df.head()

id,366367392,366367451,366367839,366367847,366368010,366368046,366368048,366368058,366368110,366368118,...,6175905346,6175905349,6175905353,6175931648,6175931660,6175931663,6175931668,6175932025,6175932032,6175932046
2009-03-27,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
2009-03-28,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
2009-03-30,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
2009-04-04,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True
2009-04-08,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,True


In [34]:
sum_bool_df = bool_df.sum(axis=1)
sum_bool_df.head()

2009-03-27    5246
2009-03-28    5246
2009-03-30    5246
2009-04-04    5246
2009-04-08    5246
dtype: int64

In [35]:
sum_bool_df[sum_bool_df == sum_bool_df.max()]

2019-01-02    10555
dtype: int64

Chọn ngày 2019-01-03 vì dữ liệu sẽ được lại tại thời điểm cập nhật trước 